In [1]:
library(keras)
library(tensorflow)
library(tidyverse)
library(recipes)
library(randomForest)
library(dplyr) 
library(xgboost)
library(caret)
library(Matrix)

Warning message:
"le package 'keras' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'tensorflow' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'tidyverse' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'ggplot2' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'tibble' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'tidyr' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'readr' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'dplyr' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'forcats' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'lubridate' a été compilé avec la version R 4.2.3"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.2     ✔ readr     2.1.4
✔ forcats   1.0.0     ✔ stringr   1.5.0
✔ ggplot2   3.4.2     ✔ tibble    3.2.1
✔ lubridate 1.9.2    

In [2]:
ConfusionMatrix <- function(y_pred, y_true) {
  Confusion_Mat <- table(y_true, y_pred)
  return(Confusion_Mat)
}
 
ConfusionDF <- function(y_pred, y_true) {
  Confusion_DF <- transform(as.data.frame(ConfusionMatrix(y_pred, y_true)),
                            y_true = as.character(y_true),
                            y_pred = as.character(y_pred),
                            Freq = as.integer(Freq))
  return(Confusion_DF)
}
 
Precision_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FP <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # it may happen that a label is never predicted (missing from y_pred) but exists in y_true
    # in this case ConfusionDF will not have these lines and thus the simplified code crashes
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]))
   
    # workaround:
    # i don't want to change ConfusionDF since i don't know if the current behaviour is a feature or a bug.
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
   
    tmp <- Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]
    FP[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Precision_micro <- sum(TP) / (sum(TP) + sum(FP))
  return(Precision_micro)
}
 
Recall_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FN <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # short version, comment out due to bug or feature of Confusion_DF
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]))
   
    # workaround:
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
 
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]
    FN[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Recall_micro <- sum(TP) / (sum(TP) + sum(FN))
  return(Recall_micro)
}
 
F1_Score_micro <- function(y_true, y_pred, labels = NULL) {
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred)) # possible problems if labels are missing from y_*
  Precision <- Precision_micro(y_true, y_pred, labels)
  Recall <- Recall_micro(y_true, y_pred, labels)
  F1_Score_micro <- 2 * (Precision * Recall) / (Precision + Recall)
  return(F1_Score_micro)
}

Let us try two strategies for the ensemble prediction. First, the most simple one would be to cast a majority vote among our 4 models.

As the 4 models perform at quite similar levels, even if some are slightly better than others, a type of voting could work. We will use a soft voting scheme, where the weighted sum of probabilities for each class is computed from the predictions of the models. 

After that, we will try to create an adaboosted version of our predictors. While this scheme is mostly used for a lot of weak learners, we will try here to use it with a few strong learners.

In [3]:
data<-read.csv("data_target_encoding.csv",stringsAsFactors = T)
test<-read.csv("test_target_encoding.csv",stringsAsFactors = T)


dataNN<-read.csv("data_target_encoding_NN.csv",stringsAsFactors = T)
testNN<-read.csv("test_target_encoding_NN.csv",stringsAsFactors = T)



data.xgb <- data
test.xgb <- test

dummy_data <- dummyVars(" ~ .", data=data.xgb)  
data.xgb <- data.frame(predict(dummy_data, newdata = data.xgb)) 
dummy_test <- dummyVars(" ~ .", data=test.xgb)  
test.xgb <- data.frame(predict(dummy_test, newdata = test.xgb)) 
   
X <- sparse.model.matrix(damage_grade ~ .,data = data.xgb)[,-1]
y <- as.numeric(data.xgb[,c("damage_grade")]-1)
test.xgb <- sparse.model.matrix( ~ .,data = test.xgb)[,-1]


In [4]:
classConverter <- function(predict_data,test_data) {
    yhat<-data.frame(matrix(0,ncol = 1, nrow = nrow(predict_data)))
    y<-data.frame(matrix(0,ncol = 1, nrow = nrow(predict_data)))
    for (i in 1:nrow(predict_data)){
        yhat[i,]<-as.integer(which.max(predict_data[i,]))
        y[i,]<-as.integer(which.max(test_data[i,])) 
    }
    mylist <- list(yhat,y)
}

In [5]:
data <- filter(data,age < 995)  #we remove outliers of age to have the same nb of rows for both

In [38]:
set.seed(2)
n_trees <- 10
nfeat <- ncol(data)-1 #I remove 1 to remove damage_grade
m_tries <- c(floor(0.5*sqrt(nfeat)))


nrounds_op <- 185
max_depth_op <- 7
eta_op <- 0.2
gamma_op <- 0.5
colsample_bytree_op <- 0.5
min_child_weight_op <- 8
subsample_op <-1


target_variable <- match('damage_grade', colnames(data))
targets<-which(grepl('damage_grade',colnames(dataNN)))



k = 3
accuracy_vec <- data.frame(matrix(0,nrow=k,ncol=4))
colnames(accuracy_vec)<-c('RF','NN','XG','Softvote')


nrows<-nrow(data)

# 1. Shuffle the dataset randomly.
data_idx <- sample(1:nrows)

# 2. Split the dataset into k groups
max <- ceiling(nrows/k)
splits <- split(data_idx, ceiling(seq_along(data_idx)/max))

pb <- txtProgressBar(min = 0, max = k, style = 3)

#3. For each unique group:
for (i in 1:k){
  #3.1 Take the group as a hold out or test data set
  test_data <- data[splits[[i]],]
  test_dataNN <- dataNN[splits[[i]],]
  #3.2 Take the remaining groups as a training data set
  train_data <- data[-splits[[i]],]   
  train_dataNN <- dataNN[-splits[[i]],]   

  model <- randomForest(x=train_data[,-c(target_variable)],
                      y=as.factor(train_data[,c(target_variable)]),
                      ntree=n_trees,mtry=m_tries,keep.forest=TRUE,importance=TRUE,type='prob')
  yhat_RF<-predict(rfmodel,test_data[,-c(target_variable)],type='prob')
  yhaty_RF <- classConverter(yhat_RF,test_dataNN[,targets])
  accuracy_vec[i,1]<-F1_Score_micro(yhaty_RF[[2]][,],yhaty_RF[[1]][,])

normalizer<-layer_normalization(axis = -1L)  %>%  
adapt(as.matrix(dataNN[,-targets]))

neuralmodel <- keras_model_sequential() %>% 
normalizer  %>% 
layer_dense(15, activation = 'relu') %>%
layer_dense(3,activation='softmax')

neuralmodel %>% compile(
    loss = 'categorical_crossentropy',
    optimizer = optimizer_adam(0.0001),
    metrics=c('AUC')
  )
  model_history <- neuralmodel %>% fit(
  as.matrix(train_dataNN[,-targets]),
  as.matrix(train_dataNN[,targets]),
  validation_split = 0.2,
  verbose = 0,
  epochs = 30
  )
  yhat_NN <- predict(neuralmodel, as.matrix(test_dataNN[-targets]))
  yhaty_NN <- classConverter(yhat_NN,test_dataNN[,targets])
  accuracy_vec[i,2]<-F1_Score_micro(yhaty_NN[[2]][,],yhaty_NN[[1]][,])


  X <- sparse.model.matrix(damage_grade ~ .,data = data.xgb[-splits[[i]],])[,-1]
  y <- as.numeric(data.xgb[-splits[[i]],c("damage_grade")]-1)
  dtrain <- xgb.DMatrix(data = X, label = y)

  bst <- xgb.train(data=dtrain,
                  max.depth=7,
                  eta=0.2,
                  nthread = 2,
                  nrounds=185,
                  objective = 'multi:softprob',
                  num_class = 3
                  )

  pred.xgb <- predict(bst, sparse.model.matrix(damage_grade ~ .,data = data.xgb[splits[[i]],])[,-1])
  yhat_XG <- data.frame(matrix(ncol=3,nrow=length(pred.xgb)/3))

  yhat_XG[,1] <- pred.xgb[seq(1,length(pred.xgb),by=3)]
  yhat_XG[,2] <- pred.xgb[seq(2,length(pred.xgb),by=3)]
  yhat_XG[,3] <- pred.xgb[seq(3,length(pred.xgb),by=3)]
  yhaty_XG <- classConverter(yhat_XG,test_dataNN[,targets])
  accuracy_vec[i,3] <- F1_Score_micro(yhaty_XG[[2]][,],yhaty_XG[[1]][,])


  yhat_softvote <- (yhat_RF + yhat_NN + yhat_XG)/3
  yhat_classvote <- data.frame(matrix(0,nrow=nrow(yhat_softvote),ncol=1))
  for (j in 1:nrow(yhat_softvote)){
    yhat_classvote[j,]<- which.max(yhat_softvote[j,]) 
  }
  
  accuracy_vec[i,4] <- F1_Score_micro(yhat_classvote[,],as.factor(test_data[,c(target_variable)]))
  rm('rfmodel','neuralmodel','bst')
  setTxtProgressBar(pb, i)
  print(accuracy_vec)

}

  |                                                                      |   0%

Majority vote

In [ ]:
set.seed(2)
n_trees <- 20
nfeat <- ncol(data)-1 #I remove 1 to remove damage_grade
m_tries <- c(floor(0.5*sqrt(nfeat)))


nrounds_op <- 185
max_depth_op <- 7
eta_op <- 0.2
gamma_op <- 0.5
colsample_bytree_op <- 0.5
min_child_weight_op <- 8
subsample_op <-1


target_variable <- match('damage_grade', colnames(data))
targets<-which(grepl('damage_grade',colnames(dataNN)))



k = 3
accuracy_vec <- data.frame(matrix(0,nrow=k,ncol=4))
colnames(accuracy_vec)<-c('RF','NN','XG','Softvote')


nrows<-nrow(data)

# 1. Shuffle the dataset randomly.
data_idx <- sample(1:nrows)

# 2. Split the dataset into k groups
max <- ceiling(nrows/k)
splits <- split(data_idx, ceiling(seq_along(data_idx)/max))

pb <- txtProgressBar(min = 0, max = k, style = 3)

#3. For each unique group:
for (i in 1:k){
  #3.1 Take the group as a hold out or test data set
  test_data <- data[splits[[i]],]
  test_dataNN <- dataNN[splits[[i]],]
  #3.2 Take the remaining groups as a training data set
  train_data <- data[-splits[[i]],]   
  train_dataNN <- dataNN[-splits[[i]],]   

  model <- randomForest(x=train_data[,-c(target_variable)],
                      y=as.factor(train_data[,c(target_variable)]),
                      ntree=n_trees,mtry=m_tries,keep.forest=TRUE,importance=TRUE)
  yhat_RF<-predict(model,test_data[,-c(target_variable)])
  accuracy_vec[i,1]<-F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat_RF)

normalizer<-layer_normalization(axis = -1L)  %>%  
adapt(as.matrix(dataNN[,-targets]))

neuralmodel <- keras_model_sequential() %>% 
normalizer  %>% 
layer_dense(37, activation = 'relu') %>%
layer_dense(3,activation='softmax')

neuralmodel %>% compile(
    loss = 'categorical_crossentropy',
    optimizer = optimizer_adam(0.0001),
    metrics=c('AUC')
  )
  model_history <- neuralmodel %>% fit(
  as.matrix(train_dataNN[,-targets]),
  as.matrix(train_dataNN[,targets]),
  validation_split = 0.2,
  verbose = 0,
  epochs = 30
  )
  yhat_NN <- predict(neuralmodel, as.matrix(test_dataNN[-targets]))
  yhaty_NN <- classConverter(yhat_NN,test_dataNN[,targets])  #first component are the predictions
  accuracy_vec[i,2]<-F1_Score_micro(yhaty_NN[[2]][,],yhaty_NN[[1]][,])


  X <- sparse.model.matrix(damage_grade ~ .,data = data.xgb[-splits[[i]],])[,-1]
  y <- as.numeric(data.xgb[-splits[[i]],c("damage_grade")]-1)
  dtrain <- xgb.DMatrix(data = X, label = y)

  bst <- xgb.train(data=dtrain,
                  max.depth=7,
                  eta=0.2,
                  nthread = 2,
                  nrounds=185,
                  objective = 'multi:softmax',
                  num_class = 3
                  )

  yhat_XGB <- predict(bst, sparse.model.matrix(damage_grade ~ .,data = data.xgb[splits[[i]],])[,-1])

  accuracy_vec[i,3] <- F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat_XGB)

  yhat_RF_votes <- recipe(~ ., yhat_RF) %>%
  step_num2factor('damage_grade',levels=c('1','2','3')) %>%
  step_dummy(all_nominal(), one_hot = TRUE) %>%
  prep(log_changes=TRUE) %>%
  bake(new_data = NULL)

  yhat_NN_votes <- recipe(~ ., yhat_NN) %>%
  step_num2factor('damage_grade',levels=c('1','2','3')) %>%
  step_dummy(all_nominal(), one_hot = TRUE) %>%
  prep(log_changes=TRUE) %>%
  bake(new_data = NULL) 

  yhat_XGB_votes <- recipe(~ ., yhat_XGB) %>%
  step_num2factor('damage_grade',levels=c('1','2','3')) %>%
  step_dummy(all_nominal(), one_hot = TRUE) %>%
  prep(log_changes=TRUE) %>%
  bake(new_data = NULL)

  yhat_votepooled <- yhat_RF_votes + yhat_NN_votes + yhat_XGB_votes
  yhat_majvoted <- data.frame(matrix(0,nrow=nrow(yhat_votepooled),ncol=1))
  for (j in 1:nrow(yhat_votepooled)){
    yhat_majovted[j,] <- which.max(yhat_votepooled[j,]) 
  }
  
  accuracy_vec[i,4] <- F1_Score_micro(yhat_majvoted[,],as.factor(test_data[,c(target_variable)]))
  rm('model','neuralmodel','bst')
  setTxtProgressBar(pb, i)
  print(accuracy_vec)

}